# MahjongMaster - treino YOLO no Google Colab

Fluxo recomendado:
1. No Colab Web, selecione **Runtime > Change runtime type > GPU**.
2. Rode as celulas em ordem.
3. Use a celula de upload se o pacote ainda nao estiver em `MyDrive/MahjongMaster/`.
4. Acompanhe o treino pelo TensorBoard dentro do notebook.


In [ ]:
!nvidia-smi
!python --version


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/MahjongMaster')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Pasta do projeto no Drive:', DRIVE_ROOT)


## Upload do dataset

Se voce ainda nao enviou o pacote, rode a celula abaixo e selecione:

`mahjongmaster_colab_dataset.zip`

Gerado localmente com:

`python scripts/prepare_colab_package.py`


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
for filename in uploaded:
    src = Path(filename)
    dest = DRIVE_ROOT / src.name
    shutil.move(str(src), dest)
    print('Enviado para:', dest)


In [ ]:
!pip install -q ultralytics tensorboard


In [ ]:
from pathlib import Path
import shutil
import yaml

PACKAGE_ZIP = DRIVE_ROOT / 'mahjongmaster_colab_dataset.zip'
WORK_ROOT = Path('/content/MahjongMaster')

assert PACKAGE_ZIP.exists(), f'Nao encontrei o pacote: {PACKAGE_ZIP}'
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

!unzip -q -o "{PACKAGE_ZIP}" -d /content/MahjongMaster

DATA_YAML = WORK_ROOT / 'data' / 'mahjong_soul_colab.yaml'
with open(WORK_ROOT / 'data' / 'mahjong_soul.yaml', 'r', encoding='utf-8') as file:
    data = yaml.safe_load(file)
data['path'] = str(WORK_ROOT / 'dataset')
with open(DATA_YAML, 'w', encoding='utf-8') as file:
    yaml.safe_dump(data, file, sort_keys=False, allow_unicode=True)

print('Dataset extraido em:', WORK_ROOT)
print('YAML Colab:', DATA_YAML)
print(DATA_YAML.read_text())


In [ ]:
from pathlib import Path

for split in ['train', 'val', 'test']:
    images = list((WORK_ROOT / 'dataset' / 'images' / split).glob('*'))
    labels = list((WORK_ROOT / 'dataset' / 'labels' / split).glob('*.txt'))
    images = [p for p in images if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]
    print(split, 'images=', len(images), 'labels=', len(labels))


## TensorBoard

Execute esta celula antes do treino. O TensorBoard vai atualizar enquanto o YOLO grava os resultados.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/MahjongMaster/runs/detect --port 6006


## Treino

Ajuste os parametros abaixo. O nome do run segue a mesma nomenclatura do app local:

`{epochs}e_{modelo}_{imgsz}p_{batch}b_colab`


In [ ]:
from ultralytics import RTDETR, YOLO
from pathlib import Path
import torch

MODEL = 'rtdetr-l.pt'
EPOCHS = 800
IMGSZ = 1600
BATCH = 5
PATIENCE = 0
RUN_NAME = f'{EPOCHS}e_{Path(MODEL).stem}_{IMGSZ}p_{BATCH}b_colab'

print('CUDA disponivel:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Run:', RUN_NAME)

model = RTDETR(MODEL) if 'rtdetr' in MODEL.lower() else YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=PATIENCE,
    project=str(WORK_ROOT / 'runs' / 'detect'),
    name=RUN_NAME,
    exist_ok=True,
)


## Copiar resultados para o Drive

Rode depois do treino. O `best.pt`, graficos e matrizes vao para `MyDrive/MahjongMaster/runs/detect/<RUN_NAME>`.


In [ ]:
RUN_DIR = WORK_ROOT / 'runs' / 'detect' / RUN_NAME
DEST_DIR = DRIVE_ROOT / 'runs' / 'detect' / RUN_NAME
DEST_DIR.parent.mkdir(parents=True, exist_ok=True)
if DEST_DIR.exists():
    shutil.rmtree(DEST_DIR)
shutil.copytree(RUN_DIR, DEST_DIR)
print('Run copiado para:', DEST_DIR)
print('best.pt:', DEST_DIR / 'weights' / 'best.pt')


In [ ]:
# Opcional: compacta o run inteiro para baixar pelo Drive.
ARCHIVE = DRIVE_ROOT / f'{RUN_NAME}.zip'
if ARCHIVE.exists():
    ARCHIVE.unlink()
shutil.make_archive(str(ARCHIVE.with_suffix('')), 'zip', DEST_DIR)
print('Arquivo zip:', ARCHIVE)
